# 04 — Virtual Screening

Screen the Enamine Diversity Set using trained pIC50 models:
1. Load best models from previous training
2. Featurize the compound library
3. Predict pIC50 with RF, XGBoost, and GNN
4. Consensus ranking
5. Filter by pIC50 threshold and Lipinski Rule of 5
6. Analyze top hits

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import joblib
import yaml
from rdkit import Chem
from rdkit.Chem import Draw
from sklearn.decomposition import PCA

from src.components.virtual_screening import VirtualScreener
from src.components.feature_engineering import MolecularFeatureEngineer
from src.components.gnn_model import EGFRGraphNet

sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
with open("../configs/config.yaml") as f:
    config = yaml.safe_load(f)

# Load trained models
rf_model = joblib.load("../models/random_forest_best.pkl")
xgb_model = joblib.load("../models/xgboost_best.pkl")

# Load GNN
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sample_graph = torch.load("../data/processed/egfr_graphs.pt", weights_only=False)[0]
num_node_features = sample_graph.x.shape[1]
gnn_model = EGFRGraphNet(
    num_node_features=num_node_features,
    hidden_channels=config["model"]["gnn"]["hidden_channels"][0],
    num_layers=config["model"]["gnn"]["num_layers"][0],
    dropout=config["model"]["gnn"]["dropout"][0],
).to(device)
gnn_model.load_state_dict(torch.load("../models/gnn_best.pt", map_location=device, weights_only=True))
gnn_model.eval()
print("Models loaded successfully")

## 1. Run Virtual Screening

In [ ]:
screener = VirtualScreener(config)

# Path to Enamine Diversity Set (update path as needed)
library_path = "../data/libraries/enamine_diversity.csv"

hits = screener.run(
    library_path=library_path,
    rf_model=rf_model,
    xgb_model=xgb_model,
    gnn_model=gnn_model,
    output_path="../results/virtual_screening_hits.csv",
)
print(f"Total hits: {len(hits)}")
hits.head(10)

## 2. Predicted pIC50 Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(hits["predicted_pIC50"], bins=30, kde=True, color="steelblue", ax=ax)
ax.axvline(x=config["virtual_screening"]["pIC50_threshold"], color="red", linestyle="--",
           label=f"Threshold (pIC50={config['virtual_screening']['pIC50_threshold']})")
ax.set_xlabel("Predicted pIC50 (Consensus)")
ax.set_ylabel("Count")
ax.set_title("Predicted pIC50 Distribution of Hits")
ax.legend()
plt.tight_layout()
plt.savefig("../results/plots/vs_pIC50_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Top Hits — 2D Structures

In [ ]:
top_10 = hits.head(10)
mols = [Chem.MolFromSmiles(smi) for smi in top_10["canonical_smiles"]]
legends = [f"pIC50={row['predicted_pIC50']:.2f}" for _, row in top_10.iterrows()]
img = Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(300, 300), legends=legends)
img

## 4. Chemical Space Overlap (Training Set vs Hits)

In [ ]:
engineer = MolecularFeatureEngineer(config)
train_df = pd.read_csv("../data/processed/egfr_curated.csv")

train_fps = engineer.compute_fingerprints(train_df["canonical_smiles"].tolist())
hits_fps = engineer.compute_fingerprints(hits["canonical_smiles"].tolist())

all_fps = np.vstack([train_fps, hits_fps])
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(all_fps)

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(coords[:len(train_fps), 0], coords[:len(train_fps), 1],
           alpha=0.3, s=10, label="Training Set", color="gray")
ax.scatter(coords[len(train_fps):, 0], coords[len(train_fps):, 1],
           alpha=0.8, s=30, label="VS Hits", color="red", edgecolors="black", linewidths=0.5)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title("Chemical Space: Training Set vs Virtual Screening Hits")
ax.legend()
plt.tight_layout()
plt.savefig("../results/plots/vs_chemical_space_overlap.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. ADMET Properties of Top Hits

In [ ]:
admet_cols = ["MolWt", "LogP", "NumHDonors", "NumHAcceptors", "Lipinski_Violations"]
available_cols = [c for c in admet_cols if c in hits.columns]
if available_cols:
    print(hits[available_cols].describe())
else:
    print("ADMET columns not found in hits dataframe — run Lipinski analysis first.")